# Anh xa probe -> gene symbol cho toan bo file selected_features_*_per_fold.csv (GSE68465)

Cac file `selected_features_*_per_fold.csv` (14 file trong
`selected_features_export/GSE68465/`) chi luu **probe set ID cua Affymetrix**
(vi du `1053_at`), khong phai ten gene, vi du du lieu duoc doc truc tiep tu
`geo_expr.h5` o muc probe (`DATA_SOURCE="geo"`, khong map gene). Notebook nay:

1. Tu dong xac dinh platform (GPL) cua GSE68465 (doc tu `pheno` trong
   `geo_expr.h5`, giong cach `_load_geo_gene_level` trong `bmo_gpu_dt.ipynb` lam).
2. Tai bang chu giai cua platform do bang `GEOparse` de lay anh xa
   `probe -> gene symbol` (tai su dung dung logic cua `_probe_to_symbol()` trong
   `GSE68465/BMO/dt/bmo_gpu_dt.ipynb`).
3. Ap dung anh xa nay len **toan bo** 14 file `selected_features_*_per_fold.csv`,
   xuat ra ban co them cot `selected_gene_symbols` (giu nguyen probe goc o cot
   `selected_features` de khong mat thong tin).

**Luu y**: mot so probe co the anh xa toi nhieu gene (vd `"TP53///TP53AS1"`) —
theo dung quy uoc da dung trong repo, notebook nay chi lay **gene dau tien**
truoc dau `///`. Mot so probe (dac biet la cac probe kiem soat `AFFX-*`) se
khong co gene symbol — cac probe nay duoc giu nguyen ID goc trong cot ket qua
va duoc dem rieng trong `n_unmapped`.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Not running in Google Colab; skipping Drive mount.")


In [ ]:
try:
    import google.colab  # noqa: F401
    get_ipython().system("pip install -q tables GEOparse")
except ModuleNotFoundError:
    pass


## 1. Xac dinh platform (GPL) cua GSE68465

In [ ]:
from pathlib import Path
import os
import pandas as pd

GSE_ID = "GSE68465"


def resolve_dataset_dir(dirname: str, required_files) -> str:
    """Tim thu muc chua du lieu da tai boi geo_lung_download.ipynb (giong het
    ham cung ten trong GSE68465/BMO/dt/bmo_gpu_dt.ipynb va notebook
    classification-comparison_with_metrics.ipynb)."""
    candidates = []

    env_dir = os.environ.get("KLTN_DATA_DIR")
    if env_dir:
        candidates.append(Path(env_dir) / dirname)

    cwd = Path.cwd()
    candidates.extend([
        cwd / "data" / dirname,
        cwd.parent / "data" / dirname,
        Path("/content/drive/MyDrive/KLTN/data") / dirname,
        Path(r"D:\KLTN\data") / dirname,
    ])

    for c in candidates:
        if all((c / f).exists() for f in required_files):
            return str(c)

    checked = "\n".join(str(c) for c in candidates)
    raise FileNotFoundError(
        f"Khong tim thay {required_files} trong bat ky thu muc nao:\n{checked}\n\n"
        "Hay chay geo_lung_download.ipynb truoc, hoac dat KLTN_DATA_DIR tro toi "
        "thu muc cha chua 'geo/'."
    )


GEO_DIR = resolve_dataset_dir("geo", ["geo_expr.h5"])
h5_path = os.path.join(GEO_DIR, "geo_expr.h5")

pheno = pd.read_hdf(h5_path, key=f"{GSE_ID}/pheno")

plats = sorted(set(pheno.get("Sample_platform_id", pd.Series(dtype=str)).dropna()))
if len(plats) != 1:
    raise RuntimeError(
        f"{GSE_ID} co {len(plats)} platform ({plats}) -- can xu ly rieng tung "
        "platform truoc khi tai bang chu giai chung."
    )
GPL_ID = plats[0]

print("GEO_DIR:", GEO_DIR)
print("Platform (GPL) cua", GSE_ID, ":", GPL_ID)


## 2. Tim thu muc chua cac file selected_features_*_per_fold.csv

In [ ]:
import json


def resolve_selected_features_dir(gse_id: str) -> Path:
    """Giong het ham cung ten trong classification-comparison_with_metrics.ipynb."""
    candidates = []

    env_dir = os.environ.get("KLTN_SELECTED_FEATURES_DIR")
    if env_dir:
        candidates.append(Path(env_dir))

    cwd = Path.cwd()
    candidates.extend([
        cwd / "selected_features_export" / gse_id,
        cwd.parent / "selected_features_export" / gse_id,
        Path("/content/drive/MyDrive/KLTN/data/selected_features") / gse_id,
        Path("/content/drive/MyDrive/selected_features_export") / gse_id,
        Path(rf"D:\KLTN\notebook\selected_features_export\{gse_id}"),
        Path(rf"D:\KLTN\notebook\{gse_id}"),  # fallback: cau truc goc long nhau
    ])

    for c in candidates:
        if not c.exists():
            continue
        if (c / "manifest.json").exists() or any(c.rglob("selected_features_*_per_fold.csv")):
            return c

    checked = "\n".join(str(c) for c in candidates)
    raise FileNotFoundError(
        f"Khong tim thay thu muc selected-features nao cho {gse_id}. Da kiem tra:\n{checked}\n\n"
        "Upload folder 'selected_features_export/GSE68465' (hoac folder GSE68465/ goc) "
        "len Drive, hoac dat KLTN_SELECTED_FEATURES_DIR tro toi thu muc chua cac file "
        "selected_features_*_per_fold.csv."
    )


def discover_fs_sources(root: Path):
    """Giong het ham cung ten trong classification-comparison_with_metrics.ipynb."""
    manifest_path = root / "manifest.json"

    if manifest_path.exists():
        with open(manifest_path, encoding="utf-8") as f:
            manifest = json.load(f)
        sources = []
        for entry in manifest:
            sel_path = root / entry["selected_features_file"]
            sources.append({
                "algorithm": entry["algorithm"],
                "fitness_classifier": entry["fitness_classifier"],
                "path": sel_path,
            })
        return sources

    sources = []
    for sel_path in sorted(root.rglob("selected_features_*_per_fold.csv")):
        d = sel_path.parent
        res_path = d / "resource_usage_summary_per_fold.csv"
        algo, clf = None, None
        if res_path.exists():
            first_row = pd.read_csv(res_path, nrows=1)
            if "algorithm" in first_row.columns:
                algo = str(first_row.loc[0, "algorithm"])
                clf = str(first_row.loc[0, "fitness_classifier"])
        if algo is None:
            algo, clf = d.parent.name, d.name
        sources.append({"algorithm": algo, "fitness_classifier": clf, "path": sel_path})
    return sources


FS_ROOT = resolve_selected_features_dir(GSE_ID)
FS_SOURCES = discover_fs_sources(FS_ROOT)

OUTPUT_DIR = FS_ROOT / "gene_symbol_mapping"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"FS_ROOT: {FS_ROOT}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"Tim thay {len(FS_SOURCES)} file selected_features can anh xa:")
for s in FS_SOURCES:
    print(f"  - {s['algorithm']:22s} | {s['fitness_classifier']:10s} | {s['path'].name}")


## 3. Tai bang chu giai cua platform va xay anh xa probe -> gene symbol

Tai su dung dung logic `_probe_to_symbol()` trong `bmo_gpu_dt.ipynb`: goi
`GEOparse.get_GEO(geo=gpl_id)` de tai bang chu giai cua platform (co cache local
o `OUTPUT_DIR/gpl_cache`, lan sau chay lai se khong tai lai), tim cot ten gene
symbol trong so cac ten cot pho bien, roi chi lay gene dau tien neu 1 probe ung
voi nhieu gene (`"A///B"` -> `"A"`).

In [ ]:
import GEOparse

SYMBOL_CANDIDATES = ["Gene Symbol", "GENE_SYMBOL", "Symbol", "gene_symbol",
                     "ILMN_Gene", "GeneSymbol", "gene_assignment"]


def fetch_probe_to_symbol(gpl_id: str, destdir: str) -> pd.DataFrame:
    """Tra ve DataFrame 2 cot [probe, gene_symbol] cho toan bo platform."""
    os.makedirs(destdir, exist_ok=True)
    gpl = GEOparse.get_GEO(geo=gpl_id, destdir=destdir, silent=True)
    tbl = gpl.table

    col = next((c for c in SYMBOL_CANDIDATES if c in tbl.columns), None)
    if col is None:
        raise RuntimeError(
            f"{gpl_id}: khong tim thay cot symbol. Cac cot co san: {list(tbl.columns)}"
        )

    m = tbl[["ID", col]].dropna().copy()
    m.columns = ["probe", "gene_symbol_raw"]
    m["gene_symbol"] = m["gene_symbol_raw"].astype(str).str.split("///").str[0].str.strip()
    m = m[~m["gene_symbol"].isin(["", "nan", "---"])]
    return m[["probe", "gene_symbol"]].drop_duplicates(subset="probe")


GPL_CACHE_DIR = OUTPUT_DIR / "gpl_cache"
probe_map_df = fetch_probe_to_symbol(GPL_ID, str(GPL_CACHE_DIR))

mapping_path = OUTPUT_DIR / f"probe_to_gene_symbol__{GPL_ID}.csv"
probe_map_df.to_csv(mapping_path, index=False, encoding="utf-8-sig")

PROBE_TO_SYMBOL = dict(zip(probe_map_df["probe"], probe_map_df["gene_symbol"]))

print(f"Platform {GPL_ID}: {len(probe_map_df)} probe co gene symbol.")
print(f"Da luu bang anh xa vao {mapping_path}")


## 4. Ap dung anh xa len toan bo file selected_features_*_per_fold.csv

In [ ]:
mapping_manifest = []

for source in FS_SOURCES:
    df = pd.read_csv(source["path"])

    mapped_col, unmapped_col = [], []
    n_mapped_col, n_unmapped_col = [], []

    for _, row in df.iterrows():
        probes = [p for p in str(row["selected_features"]).split(";") if p]

        symbols, unmapped = [], []
        for p in probes:
            sym = PROBE_TO_SYMBOL.get(p)
            if sym:
                symbols.append(sym)
            else:
                symbols.append(p)  # giu nguyen probe id neu khong anh xa duoc
                unmapped.append(p)

        mapped_col.append(";".join(symbols))
        unmapped_col.append(";".join(unmapped))
        n_mapped_col.append(len(probes) - len(unmapped))
        n_unmapped_col.append(len(unmapped))

    df["selected_gene_symbols"] = mapped_col
    df["unmapped_probes"] = unmapped_col
    df["n_mapped"] = n_mapped_col
    df["n_unmapped"] = n_unmapped_col

    out_name = source["path"].stem + "__with_genes.csv"
    out_path = OUTPUT_DIR / out_name
    df.to_csv(out_path, index=False, encoding="utf-8-sig")

    n_mapped_total = int(df["n_mapped"].sum())
    n_unmapped_total = int(df["n_unmapped"].sum())
    total = n_mapped_total + n_unmapped_total
    coverage = n_mapped_total / total if total else float("nan")

    mapping_manifest.append({
        "algorithm": source["algorithm"],
        "fitness_classifier": source["fitness_classifier"],
        "output_file": out_name,
        "n_mapped_total": n_mapped_total,
        "n_unmapped_total": n_unmapped_total,
        "coverage_ratio": coverage,
    })

    print(f"{source['algorithm']:22s} | {source['fitness_classifier']:10s} -> "
          f"{out_name} (coverage={coverage:.2%})")

manifest_df = pd.DataFrame(mapping_manifest)
manifest_df.to_csv(OUTPUT_DIR / "mapping_manifest.csv", index=False, encoding="utf-8-sig")

print(f"\nDa luu {len(mapping_manifest)} file da anh xa + mapping_manifest.csv vao {OUTPUT_DIR}")
display(manifest_df)


## Ket qua

Trong `selected_features_export/GSE68465/gene_symbol_mapping/`:
- `probe_to_gene_symbol__<GPL_ID>.csv` — bang anh xa day du probe -> gene symbol
  cua ca platform (dung lai duoc cho viec khac, khong chi 14 file nay).
- `<ten_file_goc>__with_genes.csv` — ban sao cua tung file
  `selected_features_*_per_fold.csv` goc, giu nguyen cac cot cu va them:
  - `selected_gene_symbols`: gene symbol tuong ung (cung thu tu voi
    `selected_features`; probe khong anh xa duoc thi giu nguyen probe ID).
  - `unmapped_probes`, `n_mapped`, `n_unmapped`.
- `mapping_manifest.csv` — ty le anh xa thanh cong (`coverage_ratio`) cho tung
  nguon feature-selection, de kiem tra nhanh xem co nguon nao anh xa kem bat
  thuong khong.